# Approach 1 — Query Decomposition + Facet Scoring
### Multimodal Fashion & Context Retrieval (Glance ML Assignment)

**The idea.** Vanilla CLIP encodes a whole query into *one* vector and behaves like a
"bag of concepts", so it cannot bind an attribute to the right object
("red tie + white shirt" ≈ "white tie + red shirt").

This approach **decomposes** each natural-language query into structured facets and
`(color → garment)` **bindings**, encodes each facet phrase separately with
**FashionCLIP**, scores images per-facet, and **fuses** the scores. Because each
binding ("a red tie") is scored on its own, the attribute stays glued to its
garment — directly fixing CLIP's compositionality weakness.

The notebook has two clearly-separated workflows required by the brief:
**Part A — Indexer** and **Part B — Retriever**, followed by evaluation on the 5 assessment queries.

## 0. Setup
Install dependencies (first run only) and import the modules.

In [ ]:
# !pip install -r requirements.txt   # uncomment on first run
import sys, os
sys.path.append(os.path.abspath("."))   # make `src` importable from the notebook
from src import config
print("Device:", config.get_device())
print("Images:", config.IMAGE_DIR, "->", len(list(config.IMAGE_DIR.glob('*'))), "files")

## Part A — The Indexer
Encode all 1,000 images with FashionCLIP and persist them to a **ChromaDB** vector store
(plus a NumPy matrix cache used for fast per-facet scoring). This is the offline step;
it is idempotent and cached, so re-running is instant.

In [ ]:
from src.indexer import build_index
index = build_index(force=False)   # set force=True to rebuild from scratch
print("Indexed:", len(index["ids"]), "images | embedding dim:", index["embeddings"].shape[1])

## Part B — The Retriever
### B.1 — See the decomposition
Before retrieving, inspect how a query is parsed into facets and bindings — this is the
core ML logic and the reason the system handles multi-attribute queries.

In [ ]:
from src.retriever import FacetRetriever
retriever = FacetRetriever()

import json
demo = "A red tie and a white shirt in a formal setting."
print(json.dumps(retriever.explain(demo), indent=2))

Notice `bindings` keeps *red↔tie* and *white↔shirt* as separate phrases — that is what preserves compositionality.

### B.2 — Retrieve top-k for a single query

In [ ]:
from src.evaluate import show_results
res = retriever.search("A person in a bright yellow raincoat.", k=5)
for r in res:
    print(f"{r.score:.3f}  {r.filename}  facets={ {k: round(v,2) for k,v in r.facet_scores.items()} }")
show_results("A person in a bright yellow raincoat.", res);

## Evaluation — the 5 assessment queries
Runs every required query and saves a result grid per query to `results/`.
The per-image title shows the fused score; the facet breakdown above explains *why*.

In [ ]:
from src.evaluate import run_all, EVAL_QUERIES
all_results = run_all(retriever, k=5, backend="rule", save=True)
for q, res in all_results.items():
    print("\n=>", q)
    for r in res:
        print(f"   {r.score:.3f}  {r.filename}")

### (Optional) LLM-based decomposition
Set `OPENAI_API_KEY` and pass `backend="llm"` to `search(...)` / `explain(...)` for a more
general parser. It transparently falls back to the deterministic rule-based parser if the
key is absent or the call fails, so the notebook always runs.

In [ ]:
# os.environ["OPENAI_API_KEY"] = "sk-..."
# print(json.dumps(retriever.explain(demo, backend="llm"), indent=2))

## Why this beats vanilla CLIP (summary)
- **Compositionality** — `(color→garment)` bindings are scored as standalone phrases, so
  "red tie" cannot be satisfied by a red shirt.
- **Multi-attribute queries** — garment / environment / style are separate facets, fused
  with tunable weights (`src/query_parser.py: DEFAULT_WEIGHTS`).
- **Interpretable** — every result comes with a per-facet score breakdown.
- **Zero-shot** — no training; FashionCLIP + a curated fashion vocabulary.
- **Scales** — per-facet scoring is a matmul here; at 1M images each facet becomes an ANN
  query and results are merged (see README).